In [14]:
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import gc

@dataclass
class SequenceStore:
    # Per-row arrays
    species_ids: np.ndarray
    gene_ids: np.ndarray
    sequences: list[np.ndarray]
    # Lookup tables
    species_to_gene_to_rows: dict[int, dict[int, list[int]]]
    valid_species: np.ndarray
    # Name lookups
    species_names: list[str]
    gene_names: list[str]
    species_families: np.ndarray
    # Token map used for pre-encoding
    token_to_id: dict[str, int]

    @classmethod
    def from_parquet(cls, parquet_path: str | Path, order_col: str = "order", family_col: str = "family",
                     genus_col: str = "genus", species_col: str = "species", gene_col: str = "gene", 
                     sequence_col: str = "sequence", select_orders: list[str] | None = None,
                     select_families: list[str] | None = None, select_genera: list[str] | None = None,
                     select_species: list[str] | None = None, select_genes: list[str] | None = None) -> "SequenceStore":
        print("[SequenceStore] - Loading Parquet file")
        df = pd.read_parquet(parquet_path, columns=[order_col, family_col, genus_col, species_col, gene_col,
                                                    sequence_col])

        # Filter by taxonomy or genes if provided
        for select_list, select_col in [
            (select_orders, order_col), (select_families, family_col),(select_genera, genus_col),
            (select_species, species_col), (select_genes, gene_col)]:
            if select_list is not None:
                print(f"[SequenceStore] - Filtering column {select_col} for {select_list}")
                df = df[df[select_col].isin(select_list)]
                
                # Optional: Check if any rows remain
                if df.empty:
                    raise ValueError("No rows remain after filtering")

        # Basic checks
        print("[SequenceStore] - Checking for missing values")
        if df[species_col].isna().any():
            raise ValueError(f"Column '{species_col}' contains missing values.")
        if df[gene_col].isna().any():
            raise ValueError(f"Column '{gene_col}' contains missing values.")
        if df[sequence_col].isna().any():
            raise ValueError(f"Column '{sequence_col}' contains missing values.")
        
        # Ensure categorical dtype, and remove unused categories to ensure continuous numbering
        print("[SequenceStore] - Ensuring continuous categorical species and gene columns")
        species_cat = df[species_col].astype('category').cat.remove_unused_categories()
        gene_cat = df[gene_col].astype('category').cat.remove_unused_categories()

        # Extract category integer IDs
        print("[SequenceStore] - Extracting category integer IDs")
        species_ids = species_cat.cat.codes.to_numpy(copy=True)
        gene_ids = gene_cat.cat.codes.to_numpy(copy=True)

        # Extract category names
        print("[SequenceStore] - Extracting category names")
        species_names = species_cat.cat.categories.to_list()
        gene_names = gene_cat.cat.categories.to_list()

        print("[SequenceStore] - Extracting species families")
        species_to_family = df.drop_duplicates(subset=[species_col])[[species_col, family_col]].set_index(species_col)[family_col].to_dict()
        species_families = np.array([species_to_family[name] for name in species_names])

        #Pre-encode sequences, using tiny vocabulary for DNA / ambiguous bases
        print("[SequenceStore] - Pre-encoding sequences, using tiny vocabulary for DNA / ambiguous bases ")
        token_to_id = {
            'PAD': 0,
            'A': 1,
            'C': 2,
            'G': 3,
            'T': 4,
            'N': 5,
        }

        encode_array = np.full(256, token_to_id['N'], dtype=np.uint8)
        for token, idx in token_to_id.items():
            if token != 'PAD':
                encode_array[ord(token)] = idx

        def encode_dna(seq):
        # Convert string to ASCII bytes, then use the encode_array as a lookup table
            return encode_array[np.frombuffer(seq.encode('ascii'), dtype=np.uint8)]
        
        sequences = [encode_dna(seq) for seq in df[sequence_col].array]
        
        # Build nested row-index mapping
        print("[SequenceStore] - Building nested row-index mapping")
        species_to_gene_to_rows = defaultdict(lambda: defaultdict(list))
        for row_idx, (sid, gid) in enumerate(zip(species_ids, gene_ids)):
            species_to_gene_to_rows[int(sid)][int(gid)].append(row_idx)

        species_to_gene_to_rows = {
            sid: dict(gene_map)
            for sid, gene_map in species_to_gene_to_rows.items()
        }

        valid_species = np.array(sorted(species_to_gene_to_rows.keys()))

        print("[SequenceStore] - Cleaning up temporary memory")
        del df, species_cat, gene_cat
        gc.collect()

        return cls(
            species_ids=species_ids,
            gene_ids=gene_ids,
            sequences=sequences,
            species_to_gene_to_rows=species_to_gene_to_rows,
            valid_species=valid_species,
            species_names=species_names,
            gene_names=gene_names,
            species_families=species_families,
            token_to_id=token_to_id
        )
    
    @property
    def num_rows(self) -> int:
        return len(self.sequences)
    
    @property
    def num_species(self) -> int:
        return len(self.species_names)
    
    @property
    def num_valid_species(self) -> int:
        return len(self.valid_species)
    
    @property
    def num_genes(self) -> int:
        return len(self.gene_names)
    
    @property
    def vocab_size(self) -> int:
        return len(self.token_to_id)
    
    def split_species(self, val_size: float = 0.1, test_size: float = 0.1, random_state: int = 42) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        rng = np.random.default_rng(random_state)
        
        family_to_species = defaultdict(list)
        for sid in self.valid_species:
            family = self.species_families[sid]
            family_to_species[family].append(sid)
            
        train_species = []
        val_species = []
        test_species = []
        
        for family, species_list in family_to_species.items():
            rng.shuffle(species_list)
            
            n_total = len(species_list)
            
            # For rare families (1 or 2 species), probabilistically assign the entire family to train, val, or test
            # This allows evaluating the model's zero-shot generalization to unseen families.
            if n_total < 3:
                split_choice = rng.choice(['train', 'val', 'test'], p=[1 - val_size - test_size, val_size, test_size])
                if split_choice == 'train':
                    train_species.extend(species_list)
                elif split_choice == 'val':
                    val_species.extend(species_list)
                else:
                    test_species.extend(species_list)
            else:
                n_val = int(n_total * val_size)
                n_test = int(n_total * test_size)
                n_train = n_total - n_val - n_test
                
                train_species.extend(species_list[:n_train])
                val_species.extend(species_list[n_train:n_train + n_val])
                test_species.extend(species_list[n_train + n_val:])
            
        return np.array(train_species), np.array(val_species), np.array(test_species)
    
    def summary(self) -> str:
        return (
            f"SequenceStore(num_rows={self.num_rows}, "
            f"num_species={self.num_species}, "
            f"num_genes={self.num_genes}, "
            f"valid_species={self.num_valid_species})"
        )

In [15]:
select_orders = ['Cypriniformes', 'Perciformes', 'Siluriformes', 'Gobiiformes']
select_families = None
select_genera = None
select_species = None
select_genes = None

In [16]:
path_processed_dataset = Path("output") / "processed_mitofish.parquet"
store = SequenceStore.from_parquet(path_processed_dataset, select_orders=select_orders, select_families=select_families,
                                   select_genera=select_genera, select_species=select_species,
                                   select_genes=select_genes)
store.summary()

[SequenceStore] - Loading Parquet file
[SequenceStore] - Filtering column order for ['Cypriniformes', 'Perciformes', 'Siluriformes', 'Gobiiformes']
[SequenceStore] - Checking for missing values
[SequenceStore] - Ensuring continuous categorical species and gene columns
[SequenceStore] - Extracting category integer IDs
[SequenceStore] - Extracting category names
[SequenceStore] - Extracting species families
[SequenceStore] - Pre-encoding sequences, using tiny vocabulary for DNA / ambiguous bases 
[SequenceStore] - Building nested row-index mapping
[SequenceStore] - Cleaning up temporary memory


'SequenceStore(num_rows=193241, num_species=4571, num_genes=15, valid_species=4571)'

In [17]:
@dataclass
class PhyloDist:
    species_names: list[str]
    branch_distance: np.ndarray
    topology_distance: np.ndarray


In [18]:
from dataclasses import dataclass
import numpy as np
import torch


@dataclass
class BatchBuilder:
    store: SequenceStore
    species_per_batch: int = 64
    crop_length: int = 512
    rng_seed: int | None = None
    subset_species: np.ndarray | None = None

    def __post_init__(self) -> None:
        self.rng = np.random.default_rng(self.rng_seed)
        self.pad_id = self.store.token_to_id["PAD"]
        self.available_species = self.subset_species if self.subset_species is not None else self.store.valid_species

        if self.species_per_batch < 1:
            raise ValueError("species_per_batch must be at least 1.")
        if self.crop_length < 1:
            raise ValueError("crop_length must be at least 1.")
        if self.species_per_batch > len(self.available_species):
            raise ValueError(
                "species_per_batch cannot exceed the number of available species."
            )

    @property
    def batch_size(self) -> int:
        return 2 * self.species_per_batch

    def sample_species(self) -> np.ndarray:
        return self.rng.choice(
            self.available_species,
            size=self.species_per_batch,
            replace=False,
        )

    def sample_two_rows_for_species(self, species_id: int) -> tuple[tuple[int, int], tuple[int, int]]:
        gene_to_rows = self.store.species_to_gene_to_rows[species_id]
        gene_ids = list(gene_to_rows.keys())

        # Case A: species has >= 2 genes
        if len(gene_ids) >= 2:
            gene1, gene2 = self.rng.choice(gene_ids, size=2, replace=False)
            row1 = int(self.rng.choice(gene_to_rows[gene1]))
            row2 = int(self.rng.choice(gene_to_rows[gene2]))
            return (row1, gene1), (row2, gene2)

        # Only one gene available
        gene = gene_ids[0]
        rows = gene_to_rows[gene]

        # Case B: one gene, >= 2 rows
        if len(rows) >= 2:
            row1, row2 = self.rng.choice(rows, size=2, replace=False)
            return (int(row1), gene), (int(row2), gene)

        # Case C: one gene, one row
        row = rows[0]
        return (row, gene), (row, gene)

    def write_crop_into_numpy_arrays(self, seq: np.ndarray, input_ids: np.ndarray, attention_mask: np.ndarray,
                                     batch_idx: int) -> None:
        """
        Write one cropped/padded sequence into preallocated NumPy arrays.

        Parameters
        ----------
        seq
            1D uint8 encoded sequence
        input_ids
            Preallocated NumPy array of shape [B, L], dtype=int64
        attention_mask
            Preallocated NumPy array of shape [B, L], dtype=bool
        batch_idx
            Which row in the batch to fill
        """
        L = self.crop_length
        seq_len = len(seq)

        if seq_len >= L:
            if seq_len == L:
                crop = seq
            else:
                start = self.rng.integers(0, seq_len - L + 1)
                crop = seq[start:start + L]

            input_ids[batch_idx, :] = crop
            attention_mask[batch_idx, :] = True
            return

        # Shorter than crop length: pad
        input_ids[batch_idx, :] = self.pad_id
        attention_mask[batch_idx, :] = False

        input_ids[batch_idx, :seq_len] = seq
        attention_mask[batch_idx, :seq_len] = True

    def sample_batch_cpu(self) -> dict[str, torch.Tensor]:
        """
        Build one batch on CPU using NumPy first, then convert once to torch.
        """
        sampled_species = self.sample_species()
        B = self.batch_size
        L = self.crop_length

        input_ids_np = np.full((B, L), fill_value=self.pad_id, dtype=np.int64)
        attention_mask_np = np.zeros((B, L), dtype=bool)
        species_ids_np = np.empty(B, dtype=np.int64)
        gene_ids_np = np.empty(B, dtype=np.int64)

        batch_idx = 0
        for species_id in sampled_species:
            species_id = int(species_id)
            view1, view2 = self.sample_two_rows_for_species(species_id)

            for row_idx, gene_id in (view1, view2):
                seq = self.store.sequences[row_idx]
                self.write_crop_into_numpy_arrays(
                    seq=seq,
                    input_ids=input_ids_np,
                    attention_mask=attention_mask_np,
                    batch_idx=batch_idx,
                )
                species_ids_np[batch_idx] = species_id
                gene_ids_np[batch_idx] = int(gene_id)
                batch_idx += 1

        input_ids = torch.from_numpy(input_ids_np)
        attention_mask = torch.from_numpy(attention_mask_np)
        species_ids = torch.from_numpy(species_ids_np)
        gene_ids = torch.from_numpy(gene_ids_np)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "species_ids": species_ids,
            "gene_ids": gene_ids,
        }

In [19]:
builder = BatchBuilder(
    store=store,
    species_per_batch=64,
    crop_length=512,
    rng_seed=None,
)

batch = builder.sample_batch_cpu()

for i in range(0, len(batch["species_ids"]), 2):
    sid1 = int(batch["species_ids"][i].item())
    sid2 = int(batch["species_ids"][i + 1].item())
    gid1 = int(batch["gene_ids"][i].item())
    gid2 = int(batch["gene_ids"][i + 1].item())

    print(
        f"pair {i//2}: species={store.species_names[sid1]}, "
        f"genes=({store.gene_names[gid1]}, {store.gene_names[gid2]})"
    )

pair 0: species=Scardinius acarnanicus, genes=(Cytb, COXI)
pair 1: species=Careproctus rastrinus, genes=(12S_rRNA, COXI)
pair 2: species=Leiocassis micropogon, genes=(Cytb, COXI)
pair 3: species=Hoplisoma coppenamensis, genes=(Cytb, 16S_rRNA)
pair 4: species=Knipowitschia caucasica, genes=(ND4, COXI)
pair 5: species=Labeobarbus caudovittatus, genes=(COXI, 16S_rRNA)
pair 6: species=Catostomus wigginsi, genes=(12S_rRNA, ATPase6)
pair 7: species=Acheilognathus barbatulus, genes=(ND1, ND6)
pair 8: species=Lepidocephalichthys thermalis, genes=(COXI, Cytb)
pair 9: species=Lycenchelys antarctica, genes=(COXI, COXI)
pair 10: species=Gila robusta, genes=(ATPase8, 16S_rRNA)
pair 11: species=Cetopsorhamdia iheringi, genes=(COXI, 12S_rRNA)
pair 12: species=Ballerus ballerus, genes=(Cytb, COXI)
pair 13: species=Megalonema platycephalum, genes=(COXI, 16S_rRNA)
pair 14: species=Rasbora daniconius, genes=(ATPase8, ND2)
pair 15: species=Bathyclarias worthingtoni, genes=(Cytb, Cytb)
pair 16: species=Tor

In [20]:
from torch.utils.data import IterableDataset, DataLoader, get_worker_info


class ContrastiveBatchDataset(IterableDataset):
    """
    IterableDataset that keeps your existing BatchBuilder logic and lets
    DataLoader workers prebuild full batches in the background.
    """

    def __init__(self, store: SequenceStore, species_per_batch: int = 64, crop_length: int = 512,
                 subset_species: np.ndarray | None = None, base_seed: int | None = None) -> None:
        super().__init__()
        self.store = store
        self.species_per_batch = species_per_batch
        self.crop_length = crop_length
        self.subset_species = subset_species
        self.base_seed = base_seed

    def __iter__(self):
        worker = get_worker_info()

        if worker is None:
            worker_id = 0
        else:
            worker_id = worker.id

        rng_seed = self.base_seed if self.base_seed is None else self.base_seed + worker_id
        # Each worker gets its own RNG seed, so they do not build identical batches
        builder = BatchBuilder(
            store=self.store,
            species_per_batch=self.species_per_batch,
            crop_length=self.crop_length,
            rng_seed=rng_seed,
            subset_species=self.subset_species,
        )

        while True:
            yield builder.sample_batch_cpu()


def make_batch_loader(store: SequenceStore, species_per_batch: int = 64, crop_length: int = 512,
                      subset_species: np.ndarray | None = None,
                      num_workers: int = 4, pin_memory: bool = True, prefetch_factor: int = 2,
                      persistent_workers: bool = True, base_seed: int | None = None) -> DataLoader:
    dataset = ContrastiveBatchDataset(
        store=store,
        species_per_batch=species_per_batch,
        crop_length=crop_length,
        subset_species=subset_species,
        base_seed=base_seed,
    )

    loader_kwargs = dict(
        dataset=dataset,
        batch_size=None,   # dataset already yields complete batches
        num_workers=num_workers,
        pin_memory=pin_memory,
    )

    if num_workers > 0:
        loader_kwargs["prefetch_factor"] = prefetch_factor
        loader_kwargs["persistent_workers"] = persistent_workers

    return DataLoader(**loader_kwargs)

In [21]:
import torch.nn as nn
import torch.nn.functional as F


class SequenceEncoder(nn.Module):
    """
    Transformer-based encoder for DNA sequences with optional gene embeddings.

    Input:
        input_ids:      [B, L] torch.long
        attention_mask: [B, L] torch.bool
            True  = real token
            False = padding
        gene_ids:       [B] torch.long
            gene identity for each sequence

    Output:
        z_seq: [B, embed_dim]
            L2-normalized sequence embeddings
    """

    def __init__(self, vocab_size: int, max_length: int, num_genes: int, d_model: int = 256, nhead: int = 8,
                 num_layers: int = 4, dim_feedforward: int = 1024, dropout: float = 0.1, embed_dim: int = 256,
                 pad_token_id: int = 0, use_gene_embedding: bool = True) -> None:
        super().__init__()

        if d_model % nhead != 0:
            raise ValueError("d_model must be divisible by nhead.")

        self.vocab_size = vocab_size
        self.max_length = max_length
        self.num_genes = num_genes
        self.d_model = d_model
        self.embed_dim = embed_dim
        self.pad_token_id = pad_token_id
        self.use_gene_embedding = use_gene_embedding

        # Token embedding
        self.token_embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=d_model,
            padding_idx=pad_token_id,
        )

        # Positional embedding
        self.position_embedding = nn.Embedding(
            num_embeddings=max_length,
            embedding_dim=d_model,
        )

        # Optional gene embedding
        if self.use_gene_embedding:
            self.gene_embedding = nn.Embedding(
                num_embeddings=num_genes,
                embedding_dim=d_model,
            )

        # Input dropout
        self.input_dropout = nn.Dropout(dropout)

        # Transformer encoder stack
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer=encoder_layer,
            num_layers=num_layers,
        )

        # Projection head
        self.projection = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, embed_dim),
        )

    def masked_mean_pool(self, x: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        """
        x: [B, L, d_model]
        attention_mask: [B, L] bool, True=real token, False=padding
        """
        mask = attention_mask.unsqueeze(-1).to(dtype=x.dtype)  # [B, L, 1]
        x_masked = x * mask
        summed = x_masked.sum(dim=1)  # [B, d_model]
        counts = mask.sum(dim=1).clamp(min=1.0)  # [B, 1]
        return summed / counts

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor,
                gene_ids: torch.Tensor | None = None) -> torch.Tensor:
        """
        input_ids:      [B, L] torch.long
        attention_mask: [B, L] torch.bool
        gene_ids:       [B] torch.long or None

        returns:
            z_seq: [B, embed_dim], L2-normalized
        """
        if input_ids.ndim != 2:
            raise ValueError(f"input_ids must have shape [B, L], got {input_ids.shape}")
        if attention_mask.ndim != 2:
            raise ValueError(
                f"attention_mask must have shape [B, L], got {attention_mask.shape}"
            )
        if input_ids.shape != attention_mask.shape:
            raise ValueError(
                f"input_ids and attention_mask must have same shape, got "
                f"{input_ids.shape} and {attention_mask.shape}"
            )

        batch_size, seq_len = input_ids.shape

        if seq_len > self.max_length:
            raise ValueError(
                f"Sequence length {seq_len} exceeds max_length={self.max_length}"
            )

        if self.use_gene_embedding:
            if gene_ids is None:
                raise ValueError("gene_ids must be provided when use_gene_embedding=True")
            if gene_ids.ndim != 1 or gene_ids.shape[0] != batch_size:
                raise ValueError(
                    f"gene_ids must have shape [B], got {gene_ids.shape}"
                )

        # Position indices: [B, L]
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0).expand(
            batch_size, seq_len
        )

        # Token + position embeddings
        x = self.token_embedding(input_ids) + self.position_embedding(positions)

        # Add gene embedding if enabled
        if self.use_gene_embedding:
            gene_vec = self.gene_embedding(gene_ids)          # [B, d_model]
            gene_vec = gene_vec.unsqueeze(1)                  # [B, 1, d_model]
            x = x + gene_vec                                  # broadcast over L

        # Input dropout
        x = self.input_dropout(x)

        # PyTorch transformer expects True for positions to ignore
        src_key_padding_mask = ~attention_mask  # [B, L]

        x = self.transformer(x, src_key_padding_mask=src_key_padding_mask)  # [B, L, d_model]

        pooled = self.masked_mean_pool(x, attention_mask)    # [B, d_model]
        z_seq = self.projection(pooled)                      # [B, embed_dim]
        z_seq = F.normalize(z_seq, dim=-1)

        return z_seq

In [22]:
class SpeciesContrastiveModel(nn.Module):
    """
    Sequence encoder + species prototype table.

    Training objective:
        - batch-restricted prototype loss
        - half-vs-half CLIP-style sequence contrastive loss
    """

    def __init__(self, encoder: SequenceEncoder, num_species: int, embed_dim: int, temperature: float = 0.07,
                 lambda_proto: float = 1.0, lambda_seq: float = 1.0) -> None:
        super().__init__()

        if temperature <= 0:
            raise ValueError("temperature must be > 0")

        self.encoder = encoder
        self.temperature = temperature
        self.lambda_proto = lambda_proto
        self.lambda_seq = lambda_seq

        self.species_embedding = nn.Embedding(num_species, embed_dim)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor, species_ids: torch.Tensor,
                gene_ids: torch.Tensor) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
        """
        Loss = batch-restricted prototype loss + half-vs-half sequence contrastive loss

        Assumes the batch is organized in paired views:
            0,1   = same species
            2,3   = same species
            4,5   = same species
            ...

        Parameters
        ----------
        input_ids : [B, L]
        attention_mask : [B, L]
        species_ids : [B]
        gene_ids : [B]

        Returns
        -------
        total_loss : scalar tensor
        outputs : dict of tensors for logging/debugging
        """
        # ------------------------------------------------------------
        # 1) Encode sequences
        # ------------------------------------------------------------
        z_seq = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            gene_ids=gene_ids,
        )  # [B, D], already normalized by encoder

        B = z_seq.shape[0]
        device = z_seq.device

        if B % 2 != 0:
            raise ValueError(f"Batch size must be even, got {B}")

        # ------------------------------------------------------------
        # 2) Batch-restricted prototype loss
        # ------------------------------------------------------------
        batch_species_ids, local_targets = torch.unique(
            species_ids,
            sorted=True,
            return_inverse=True,
        )  # batch_species_ids: [S], local_targets: [B]

        z_species_batch = self.species_embedding(batch_species_ids)  # [S, D]
        z_species_batch = F.normalize(z_species_batch, dim=-1)

        logits_proto = (z_seq @ z_species_batch.T) / self.temperature  # [B, S]
        loss_proto = F.cross_entropy(logits_proto, local_targets)

        # ------------------------------------------------------------
        # 3) Half-vs-half CLIP-style sequence contrastive loss
        # ------------------------------------------------------------
        # Split into the two views
        z_a = z_seq[0::2]   # [S, D]
        z_b = z_seq[1::2]   # [S, D]

        S = z_a.shape[0]
        targets_seq = torch.arange(S, device=device)  # diagonal targets

        logits_seq = (z_a @ z_b.T) / self.temperature  # [S, S]

        # Symmetric CLIP-style loss
        loss_seq_ab = F.cross_entropy(logits_seq, targets_seq)
        loss_seq_ba = F.cross_entropy(logits_seq.T, targets_seq)
        loss_seq = 0.5 * (loss_seq_ab + loss_seq_ba)

        # NOTE: a phylogenetic-tree-aware KL loss was previously sketched here
        # but referenced attributes (`self.phylo_temp_embed`, `self.phylo_temp_tree`,
        # `self.tree_distance_matrix`) and an undefined local `sim` that were
        # never wired up. It has been removed so the forward pass runs cleanly.
        # Re-introduce it as a separate, fully-initialized `PhyloAwareSpeciesContrastiveModel`
        # subclass (or a new mixin) when the tree-distance matrix becomes available.

        # ------------------------------------------------------------
        # 4) Combine losses
        # ------------------------------------------------------------
        total_loss = self.lambda_proto * loss_proto + self.lambda_seq * loss_seq

        outputs = {
            "z_seq": z_seq,
            "batch_species_ids": batch_species_ids,   # [S]
            "z_species_batch": z_species_batch,       # [S, D]
            "logits_proto": logits_proto,             # [B, S]
            "logits_seq": logits_seq,                 # [S, S]
            "loss_proto": loss_proto.detach(),
            "loss_seq": loss_seq.detach(),
            "local_targets": local_targets,           # [B]
            "targets_seq": targets_seq,               # [S]
        }

        return total_loss, outputs

In [23]:
import time
from pathlib import Path

import torch


def topk_accuracy(logits: torch.Tensor, targets: torch.Tensor, k: int = 1) -> torch.Tensor:
    """
    Compute top-k accuracy.

    logits:  [N, C]
    targets: [N]
    """
    k = min(k, logits.shape[1])
    topk = logits.topk(k, dim=1).indices
    correct = (topk == targets.unsqueeze(1)).any(dim=1)
    return correct.float().mean()


def save_checkpoint(save_path: str | Path, model: SpeciesContrastiveModel, optimizer: torch.optim.Optimizer,
                    step: int, loss: float, loss_proto: float, loss_seq: float, acc_proto_top1: float,
                    acc_proto_top5: float, acc_seq: float, species_names: list[str]) -> None:
    """
    Save a training checkpoint including model weights and the current
    species embedding matrix for downstream clustering analysis.
    """
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)

    checkpoint = {
        "step": step,
        "loss": loss,
        "loss_proto": loss_proto,
        "loss_seq": loss_seq,
        "acc_proto_top1": acc_proto_top1,
        "acc_proto_top5": acc_proto_top5,
        "acc_seq": acc_seq,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "species_embeddings": model.species_embedding.weight.detach().cpu(),
        "species_names": species_names,
    }

    torch.save(checkpoint, save_path)

def move_batch_to_device(batch: dict[str, torch.Tensor], device: str | torch.device,
                         non_blocking: bool = True) -> dict[str, torch.Tensor]:
    device = torch.device(device)
    return {
        k: v.to(device, non_blocking=non_blocking)
        for k, v in batch.items()
    }

def evaluate_species_contrastive(model: SpeciesContrastiveModel, loader: DataLoader,
                                 device: str | torch.device, eval_steps: int = 100,
                                 use_amp: bool = True) -> dict[str, float]:
    """
    Evaluate the species-contrastive model over a fixed number of steps.
    """
    device = torch.device(device)
    amp_enabled = use_amp and device.type == "cuda"
    
    model.eval()
    
    total_loss = 0.0
    total_loss_proto = 0.0
    total_loss_seq = 0.0
    total_acc_proto_top1 = 0.0
    total_acc_proto_top5 = 0.0
    total_acc_seq = 0.0
    
    batch_iter = iter(loader)
    with torch.no_grad():
        for step in range(eval_steps):
            batch_cpu = next(batch_iter)
            batch = move_batch_to_device(batch_cpu, device=device, non_blocking=True)
            
            with torch.autocast("cuda", enabled=amp_enabled):
                loss, outputs = model(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                    species_ids=batch["species_ids"],
                    gene_ids=batch["gene_ids"],
                )
                
            logits_proto = outputs["logits_proto"]
            logits_seq = outputs["logits_seq"]
            local_targets = outputs["local_targets"]
            targets_seq = outputs["targets_seq"]
            
            acc_proto_top1 = topk_accuracy(logits_proto.float(), local_targets, k=1)
            acc_proto_top5 = topk_accuracy(logits_proto.float(), local_targets, k=5)
            acc_seq = topk_accuracy(logits_seq.float(), targets_seq, k=1)
            
            total_loss += float(loss.item())
            total_loss_proto += float(outputs["loss_proto"].item())
            total_loss_seq += float(outputs["loss_seq"].item())
            total_acc_proto_top1 += float(acc_proto_top1.item())
            total_acc_proto_top5 += float(acc_proto_top5.item())
            total_acc_seq += float(acc_seq.item())
            
    return {
        "val_loss": total_loss / eval_steps,
        "val_loss_proto": total_loss_proto / eval_steps,
        "val_loss_seq": total_loss_seq / eval_steps,
        "val_acc_proto_top1": total_acc_proto_top1 / eval_steps,
        "val_acc_proto_top5": total_acc_proto_top5 / eval_steps,
        "val_acc_seq": total_acc_seq / eval_steps,
    }

def train_species_contrastive(model: SpeciesContrastiveModel, loader: DataLoader, optimizer: torch.optim.Optimizer,
                              device: str | torch.device, num_steps: int, checkpoint_dir: str | Path,
                              checkpoint_every: int = 500, log_every: int = 50, grad_clip_norm: float | None = 1.0,
                              start_step: int = 0, use_amp: bool = True,
                              species_names: list[str] | None = None) -> list[dict]:
    """
    Train the species-contrastive model with optional AMP.

    Periodically saves checkpoints containing:
    - encoder + species embedding weights
    - optimizer state
    - current species embedding matrix
    - metadata (step, losses, accuracies)

    Returns
    -------
    history : list of dict
        Per-log-step training metrics.
    """
    device = torch.device(device)
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    amp_enabled = use_amp and device.type == "cuda"
    scaler = torch.GradScaler("cuda", enabled=amp_enabled)

    model.train()
    history = []

    t0 = time.time()
    batch_iter = iter(loader)

    for step in range(start_step + 1, start_step + num_steps + 1):
        # Build batch on CPU, then move once to GPU
        batch_cpu = next(batch_iter)
        batch = move_batch_to_device(batch_cpu, device=device, non_blocking=True)

        with torch.autocast("cuda", enabled=amp_enabled):
            loss, outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                species_ids=batch["species_ids"],
                gene_ids=batch["gene_ids"],
            )

        logits_proto = outputs["logits_proto"]   # [B, S]
        logits_seq = outputs["logits_seq"]       # [S, S]
        local_targets = outputs["local_targets"] # [B]
        targets_seq = outputs["targets_seq"]     # [S]

        # Metrics are computed outside autocast for consistency/readability
        acc_proto_top1 = topk_accuracy(logits_proto.float(), local_targets, k=1)
        acc_proto_top5 = topk_accuracy(logits_proto.float(), local_targets, k=5)
        acc_seq = topk_accuracy(logits_seq.float(), targets_seq, k=1)

        loss_proto = outputs["loss_proto"]
        loss_seq = outputs["loss_seq"]

        optimizer.zero_grad(set_to_none=True)

        if amp_enabled:
            scaler.scale(loss).backward()

            if grad_clip_norm is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)

            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()

            if grad_clip_norm is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)

            optimizer.step()

        # Logging
        if step % log_every == 0 or step == start_step + 1:
            elapsed = time.time() - t0
            metrics = {
                "step": step,
                "loss": float(loss.item()),
                "loss_proto": float(loss_proto.item()),
                "loss_seq": float(loss_seq.item()),
                "acc_proto_top1": float(acc_proto_top1.item()),
                "acc_proto_top5": float(acc_proto_top5.item()),
                "acc_seq": float(acc_seq.item()),
                "elapsed_sec": elapsed,
            }
            history.append(metrics)

            print(
                f"[step {step:6d}] "
                f"loss={metrics['loss']:.4f} "
                f"proto={metrics['loss_proto']:.4f} "
                f"seq={metrics['loss_seq']:.4f} "
                f"acc1={metrics['acc_proto_top1']:.4f} "
                f"acc5={metrics['acc_proto_top5']:.4f} "
                f"acc_seq={metrics['acc_seq']:.4f} "
                f"time={elapsed:.1f}s"
            )

        # Periodic checkpointing
        if step % checkpoint_every == 0 or step == start_step + num_steps:
            save_path = checkpoint_dir / f"step_{step:07d}.pt"
            save_checkpoint(
                save_path=save_path,
                model=model,
                optimizer=optimizer,
                step=step,
                loss=float(loss.item()),
                loss_proto=float(loss_proto.item()),
                loss_seq=float(loss_seq.item()),
                acc_proto_top1=float(acc_proto_top1.item()),
                acc_proto_top5=float(acc_proto_top5.item()),
                acc_seq=float(acc_seq.item()),
                species_names=species_names if species_names is not None else [],
            )
            print(f"Saved checkpoint to {save_path}")

    return history

In [24]:
checkpoint_dir: str = "output/checkpoints_species_contrastive_top4orders_b128"
resume_ckpt_path: str | None = "output/checkpoints_species_contrastive_top4orders_b64/step_2000000.pt"
num_steps: int = 500_000
species_per_batch: int = 128
crop_length: int = 512

In [25]:
"""
CUDA smoke test.

Run this once after (re)starting the kernel to confirm:
  - PyTorch is the CUDA build
  - the RTX 3070 is visible
  - a tiny end-to-end forward pass on the SequenceContrastiveModel succeeds on GPU

If any of these fail, do not start the Optuna search below -- the search would
just OOM-kill the kernel like before.
"""
import torch

print(f"torch:        {torch.__version__}")
print(f"cuda built:   {torch.version.cuda}")
print(f"cuda avail:   {torch.cuda.is_available()}")

assert torch.cuda.is_available(), (
    "CUDA is not available. Reinstall the CUDA build of PyTorch:\n"
    "  pip install --upgrade --index-url https://download.pytorch.org/whl/cu126 "
    "torch==2.7.0 torchvision torchaudio"
)

device = torch.device("cuda")
props = torch.cuda.get_device_properties(0)
print(f"  GPU:         {props.name}")
print(f"  VRAM:        {props.total_memory / 2**30:.1f} GiB")
print(f"  Compute cap: {props.major}.{props.minor}")

# Tiny model + tiny batch on GPU to exercise the full forward pass.
_smoke_encoder = SequenceEncoder(
    vocab_size=store.vocab_size,
    max_length=128,
    num_genes=store.num_genes,
    d_model=64,
    nhead=4,
    num_layers=2,
    dim_feedforward=128,
    dropout=0.0,
    embed_dim=64,
    pad_token_id=store.token_to_id["PAD"],
    use_gene_embedding=True,
).to(device)

_smoke_model = SpeciesContrastiveModel(
    encoder=_smoke_encoder,
    num_species=store.num_valid_species,
    embed_dim=64,
    temperature=0.1,
).to(device)

_smoke_input_ids = torch.randint(
    0, store.vocab_size, size=(8, 128), device=device, dtype=torch.long
)
_smoke_mask = torch.ones((8, 128), device=device, dtype=torch.bool)
_smoke_species = torch.randint(
    0, store.num_valid_species, size=(8,), device=device, dtype=torch.long
)
_smoke_genes = torch.randint(
    0, store.num_genes, size=(8,), device=device, dtype=torch.long
)

with torch.autocast("cuda", dtype=torch.float16):
    _loss, _ = _smoke_model(
        input_ids=_smoke_input_ids,
        attention_mask=_smoke_mask,
        species_ids=_smoke_species,
        gene_ids=_smoke_genes,
    )
_loss.backward()

print(f"\nForward+backward on GPU OK (loss={_loss.item():.4f}).")
print(f"VRAM after smoke test: "
      f"{torch.cuda.memory_allocated() / 2**20:.1f} MiB allocated, "
      f"{torch.cuda.memory_reserved() / 2**20:.1f} MiB reserved.")

del _smoke_model, _smoke_encoder, _smoke_input_ids, _smoke_mask, _smoke_species, _smoke_genes, _loss
torch.cuda.empty_cache()

torch:        2.7.0+cu126
cuda built:   12.6
cuda avail:   True
  GPU:         NVIDIA GeForce RTX 3070 Laptop GPU
  VRAM:        8.0 GiB
  Compute cap: 8.6

Forward+backward on GPU OK (loss=4.6685).
VRAM after smoke test: 19.6 MiB allocated, 48.0 MiB reserved.


/home/nadiantara/miniconda3/envs/pytorch_env/lib/python3.10/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


In [ ]:
import gc
from pathlib import Path

import optuna
import torch
from optuna.pruners import HyperbandPruner
from optuna.samplers import TPESampler

# ----------------------------------------------------------------
# 1. Device + GPU sanity check
# ----------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_cuda = device.type == "cuda"

print(f"Device: {device}")
if use_cuda:
    props = torch.cuda.get_device_properties(0)
    print(f"  GPU: {props.name} ({props.total_memory / 2**30:.1f} GiB VRAM)")
else:
    print("  WARNING: CUDA not available -- this notebook is sized for an 8 GiB GPU.")
    print("  On CPU each trial takes hours; consider running only a smoke-test (n_trials=1).")

# ----------------------------------------------------------------
# 2. Species splits
# ----------------------------------------------------------------
train_species, val_species, test_species = store.split_species(
    val_size=0.1, test_size=0.1, random_state=42
)
print(f"Train species: {len(train_species)}")
print(f"Val species:   {len(val_species)}")
print(f"Test species:  {len(test_species)}")

# ----------------------------------------------------------------
# 3. Data loaders -- sized for the actual hardware
# ----------------------------------------------------------------
# species_per_batch is intentionally smaller during the search than during the
# final training run, so the largest sampled trial still fits in 8 GiB VRAM
# with AMP. Increase it after the best HPs are picked.
hp_species_per_batch = 64 if use_cuda else 8
hp_crop_length = crop_length  # 512 from the upstream parameters cell

train_loader = make_batch_loader(
    store=store,
    species_per_batch=hp_species_per_batch,
    crop_length=hp_crop_length,
    subset_species=train_species,
    num_workers=4 if use_cuda else 0,
    pin_memory=use_cuda,
    prefetch_factor=2 if use_cuda else None,
    persistent_workers=use_cuda,
    base_seed=None,
)

val_loader = make_batch_loader(
    store=store,
    species_per_batch=hp_species_per_batch,
    crop_length=hp_crop_length,
    subset_species=val_species,
    num_workers=2 if use_cuda else 0,
    pin_memory=use_cuda,
    prefetch_factor=2 if use_cuda else None,
    persistent_workers=False,
    base_seed=None,
)

# Note: test_loader is intentionally NOT built here -- the test set must not
# influence hyperparameter selection. Build it after the best trial is chosen.

# ----------------------------------------------------------------
# 4. Optuna study: persistent SQLite + Hyperband pruner
# ----------------------------------------------------------------
optuna_dir = Path("output/optuna")
optuna_dir.mkdir(parents=True, exist_ok=True)
storage = f"sqlite:///{(optuna_dir / 'stage1_transformer_stratified.db').as_posix()}"

# Each trial trains for at most MAX_RESOURCE * EVAL_EVERY steps, with a
# validation pass every EVAL_EVERY steps. Hyperband prunes bad trials at
# rungs of (1, 2, 4) * EVAL_EVERY, so most trials finish at the cheapest rung.
EVAL_EVERY: int   = 500
MIN_RESOURCE: int = 1   # 500  train steps
MAX_RESOURCE: int = 4   # 2000 train steps for the longest survivors

study = optuna.create_study(
    study_name="stage1_transformer_stratified",
    storage=storage,
    load_if_exists=True,
    direction="minimize",
    sampler=TPESampler(seed=42, n_startup_trials=8, multivariate=True, group=True),
    pruner=HyperbandPruner(
        min_resource=MIN_RESOURCE,
        max_resource=MAX_RESOURCE,
        reduction_factor=3,
    ),
)

# ----------------------------------------------------------------
# 5. Objective
# ----------------------------------------------------------------
def objective(trial: optuna.Trial) -> float:
    # Search space -- bounded so the largest trial fits in 8 GiB VRAM with AMP.
    d_model         = trial.suggest_categorical("d_model", [128, 256])
    nhead           = trial.suggest_categorical("nhead", [4, 8])
    num_layers      = trial.suggest_int("num_layers", 2, 4)
    dim_feedforward = trial.suggest_categorical("dim_feedforward", [512, 1024])
    dropout         = trial.suggest_float("dropout", 0.0, 0.3)
    lr              = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    temperature     = trial.suggest_float("temperature", 0.05, 0.2)

    if d_model % nhead != 0:
        raise optuna.TrialPruned()

    encoder = SequenceEncoder(
        vocab_size=store.vocab_size,
        max_length=hp_crop_length,
        num_genes=store.num_genes,
        d_model=d_model,
        nhead=nhead,
        num_layers=num_layers,
        dim_feedforward=dim_feedforward,
        dropout=dropout,
        embed_dim=256,
        pad_token_id=store.token_to_id["PAD"],
        use_gene_embedding=True,
    ).to(device)

    model = SpeciesContrastiveModel(
        encoder=encoder,
        num_species=store.num_valid_species,
        embed_dim=256,
        temperature=temperature,
        lambda_proto=1.0,
        lambda_seq=1.0,
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=lr, weight_decay=1e-2
    )

    val_loss = float("inf")

    try:
        for resource in range(1, MAX_RESOURCE + 1):
            start_step = (resource - 1) * EVAL_EVERY

            train_species_contrastive(
                model=model,
                loader=train_loader,
                optimizer=optimizer,
                device=device,
                num_steps=EVAL_EVERY,
                # Checkpoint dir is per-trial; checkpoint_every is set very high
                # so only the end-of-window checkpoint is written. Disk usage is
                # bounded to ~1 file per resource rung per surviving trial.
                checkpoint_dir=optuna_dir / f"trial_{trial.number:04d}",
                checkpoint_every=10**9,
                log_every=200,
                grad_clip_norm=1.0,
                start_step=start_step,
                use_amp=use_cuda,
                species_names=store.species_names,
            )

            val_metrics = evaluate_species_contrastive(
                model, val_loader, device,
                eval_steps=50,
                use_amp=use_cuda,
            )
            val_loss = float(val_metrics["val_loss"])

            # Persist the full eval breakdown (val_loss_proto, val_loss_seq,
            # val_acc_proto_top1/top5, val_acc_seq) on the trial so it shows
            # up as `user_attrs_<key>` columns in `study.trials_dataframe()`.
            # Each rung overwrites the previous one, leaving the latest
            # snapshot at trial end. `last_resource` records which rung that was.
            for _k, _v in val_metrics.items():
                trial.set_user_attr(_k, float(_v))
            trial.set_user_attr("last_resource", resource)

            trial.report(val_loss, step=resource)
            if trial.should_prune():
                raise optuna.TrialPruned()

        return val_loss

    except torch.cuda.OutOfMemoryError:
        # An unlucky combination of hyperparameters spilled over VRAM --
        # mark the trial as pruned rather than failed so the search continues.
        if use_cuda:
            torch.cuda.empty_cache()
        raise optuna.TrialPruned()

    finally:
        del model, encoder, optimizer
        gc.collect()
        if use_cuda:
            torch.cuda.empty_cache()


# ----------------------------------------------------------------
# 6. Run the study
# ----------------------------------------------------------------
study.optimize(
    objective,
    n_trials=30,
    timeout=3600 * 6,        # 6 h wall-clock budget; remove or raise as desired
    gc_after_trial=True,
    show_progress_bar=True,
    catch=(RuntimeError,),   # don't kill the study on a single trial's transient error
)

print("\nBest trial:")
print(f"  number: {study.best_trial.number}")
print(f"  value:  {study.best_trial.value:.4f}")
print(f"  params: {study.best_trial.params}")
print(f"\nStudy DB: {storage}")
print(f"  Resume / inspect later with:  optuna.load_study(study_name=..., storage=...)")

Device: cuda
  GPU: NVIDIA GeForce RTX 3070 Laptop GPU (8.0 GiB VRAM)
Train species: 3736
Val species:   414
Test species:  421


/home/nadiantara/miniconda3/envs/pytorch_env/lib/python3.10/site-packages/optuna/_experimental.py:33: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  optuna_warn(
/home/nadiantara/miniconda3/envs/pytorch_env/lib/python3.10/site-packages/optuna/_experimental.py:33: ExperimentalWarning: Argument ``group`` is an experimental feature. The interface can change in the future.
  optuna_warn(
[I 2026-05-01 07:33:56,110] Using an existing study with name 'stage1_transformer_stratified' instead of creating a new one.


  0%|          | 0/30 [00:00<?, ?it/s]

[step      1] loss=9.1146 proto=4.2130 seq=4.9016 acc1=0.0078 acc5=0.0859 acc_seq=0.0000 time=0.9s
[step    200] loss=8.2285 proto=4.1991 seq=4.0294 acc1=0.0234 acc5=0.0859 acc_seq=0.0312 time=29.1s
[step    400] loss=8.1377 proto=4.1881 seq=3.9496 acc1=0.0000 acc5=0.0781 acc_seq=0.0625 time=57.6s
Saved checkpoint to output/optuna/trial_0014/step_0000500.pt
[step    501] loss=8.1943 proto=4.1618 seq=4.0325 acc1=0.0312 acc5=0.1016 acc_seq=0.0000 time=0.1s
[step    600] loss=8.1158 proto=4.1718 seq=3.9440 acc1=0.0078 acc5=0.0859 acc_seq=0.0000 time=14.1s
[step    800] loss=8.1029 proto=4.1490 seq=3.9538 acc1=0.0469 acc5=0.1016 acc_seq=0.0156 time=42.3s
[step   1000] loss=8.2570 proto=4.1626 seq=4.0945 acc1=0.0312 acc5=0.1328 acc_seq=0.0312 time=70.7s
Saved checkpoint to output/optuna/trial_0014/step_0001000.pt
[step   1001] loss=7.9694 proto=4.1265 seq=3.8429 acc1=0.0312 acc5=0.1328 acc_seq=0.0312 time=0.2s
[step   1200] loss=8.0228 proto=4.1170 seq=3.9059 acc1=0.0234 acc5=0.1484 acc_seq

In [ ]:
"""
Backfill per-metric eval breakdown for already-completed trials.

The original objective only stored `value` (= val_loss total) per trial, so
val_loss_proto / val_loss_seq / val_acc_* were computed during training but
discarded. Each surviving trial did, however, save model checkpoints at every
Hyperband rung end (`output/optuna/trial_<NNNN>/step_<step>.pt`). This cell:

  1. Walks every COMPLETE trial in the study.
  2. Reconstructs the model from the trial's params.
  3. Loads the latest (= longest-trained) saved checkpoint.
  4. Re-runs `evaluate_species_contrastive` on the same val_loader.
  5. Writes the full metric breakdown to a sidecar `trials_recovery.csv`.

Optuna 4.x raises `UpdateFinishedTrialError` if you try to write back to a
completed trial's `user_attrs`, so we keep the recovered metrics in a sidecar
CSV. The export cell below left-joins it into the final `trials.csv`. For
*future* trials, the objective uses `trial.set_user_attr` during the trial
itself (writes are allowed mid-trial), so the breakdown shows up natively in
`study.trials_dataframe()` without going through this recovery step.

Run-once after a search completes; safe to re-run (idempotent overwrite).
Requires CUDA (skips with a warning otherwise).
"""
import re
from pathlib import Path

import optuna
import pandas as pd
import torch

OPTUNA_DIR = Path("output/optuna")
STUDY_NAME = "stage1_transformer_stratified"
STORAGE = f"sqlite:///{(OPTUNA_DIR / f'{STUDY_NAME}.db').as_posix()}"

study = optuna.load_study(study_name=STUDY_NAME, storage=STORAGE)

if not torch.cuda.is_available():
    print("CUDA not available -- skipping recovery (re-evaluating ~16 transformer "
          "trials on CPU is impractical). Re-run after enabling GPU.")
else:
    device = torch.device("cuda")
    recovery_rows: list[dict] = []
    skipped: list[tuple[int, str]] = []

    for trial in study.trials:
        if trial.state.name != "COMPLETE":
            continue

        trial_dir = OPTUNA_DIR / f"trial_{trial.number:04d}"
        ckpts = sorted(trial_dir.glob("step_*.pt")) if trial_dir.exists() else []
        if not ckpts:
            skipped.append((trial.number, "no checkpoint"))
            continue

        latest_ckpt = ckpts[-1]
        ckpt_step = int(re.search(r"step_(\d+)", latest_ckpt.name).group(1))
        p = trial.params

        encoder = SequenceEncoder(
            vocab_size=store.vocab_size,
            max_length=hp_crop_length,
            num_genes=store.num_genes,
            d_model=p["d_model"],
            nhead=p["nhead"],
            num_layers=p["num_layers"],
            dim_feedforward=p["dim_feedforward"],
            dropout=p["dropout"],
            embed_dim=256,
            pad_token_id=store.token_to_id["PAD"],
            use_gene_embedding=True,
        ).to(device)
        model = SpeciesContrastiveModel(
            encoder=encoder,
            num_species=store.num_valid_species,
            embed_dim=256,
            temperature=p["temperature"],
            lambda_proto=1.0,
            lambda_seq=1.0,
        ).to(device)

        try:
            ckpt = torch.load(latest_ckpt, map_location=device, weights_only=False)
            model.load_state_dict(ckpt["model_state_dict"])

            val_metrics = evaluate_species_contrastive(
                model, val_loader, device, eval_steps=50, use_amp=True
            )
        except Exception as exc:
            skipped.append((trial.number, f"{type(exc).__name__}: {exc}"))
            del model, encoder
            torch.cuda.empty_cache()
            continue

        recovery_rows.append({
            "number": trial.number,
            "recovered_from_step": ckpt_step,
            **{k: float(v) for k, v in val_metrics.items()},
        })

        print(
            f"  trial #{trial.number:>3d}  step {ckpt_step:>5d}  "
            f"val_loss={val_metrics['val_loss']:.4f} "
            f"proto={val_metrics['val_loss_proto']:.4f} "
            f"seq={val_metrics['val_loss_seq']:.4f} "
            f"acc1={val_metrics['val_acc_proto_top1']:.3f} "
            f"acc5={val_metrics['val_acc_proto_top5']:.3f} "
            f"acc_seq={val_metrics['val_acc_seq']:.3f}"
        )

        del model, encoder, ckpt
        torch.cuda.empty_cache()

    if recovery_rows:
        recovery_df = pd.DataFrame(recovery_rows).sort_values("number").reset_index(drop=True)
        recovery_csv = OPTUNA_DIR / "trials_recovery.csv"
        recovery_df.to_csv(recovery_csv, index=False)
        print(f"\nWrote {recovery_csv}  ({len(recovery_df)} trials backfilled)")

    if skipped:
        print("\nSkipped trials:")
        for n, why in skipped:
            print(f"  #{n}: {why}")

In [27]:
"""
Post-search analysis & artifact export.

Run this cell AFTER the Optuna `study.optimize(...)` call above has returned.

The SQLite store at output/optuna/stage1_transformer_stratified.db is the
authoritative record -- this cell just unpacks it into convenient files for
sharing, plotting, and the next-stage final retrain.

Outputs (under output/optuna/):
  trials.csv               full per-trial table (params + value + state + timing)
  trials_intermediate.csv  per-rung val_loss reports for every trial
  best_params.json         study.best_trial.params + value/number for retrain
  history.html             optimization-history plot
  intermediate_values.html per-rung val_loss curves
  parallel_coordinate.html params <-> objective parallel coordinate plot
  slice.html               per-param slice plots
  param_importances.html   hyperparameter importance bar chart (needs sklearn)
"""
import json
from collections import Counter
from pathlib import Path

import optuna
import optuna.visualization as ov
import pandas as pd

OPTUNA_DIR = Path("output/optuna")
STUDY_NAME = "stage1_transformer_stratified"
STORAGE = f"sqlite:///{(OPTUNA_DIR / f'{STUDY_NAME}.db').as_posix()}"

# Read-only load: safe to run while other notebooks/readers are open.
study = optuna.load_study(study_name=STUDY_NAME, storage=STORAGE)

# ------------------------------------------------------------------
# 1. Per-trial table -- one row per trial.
# ------------------------------------------------------------------
trials_df = study.trials_dataframe(
    attrs=(
        "number", "value", "state", "params", "user_attrs",
        "datetime_start", "datetime_complete", "duration",
    ),
    multi_index=False,
)
if "duration" in trials_df.columns:
    trials_df["duration_seconds"] = (
        pd.to_timedelta(trials_df["duration"]).dt.total_seconds()
    )

# Left-join the sidecar recovery CSV (if the recovery cell has been run).
# This adds val_loss_proto / val_loss_seq / val_acc_* columns for trials that
# completed *before* the objective started writing user_attrs, so the final
# `trials.csv` is one unified view regardless of when each trial finished.
recovery_csv = OPTUNA_DIR / "trials_recovery.csv"
if recovery_csv.exists():
    rec = pd.read_csv(recovery_csv)
    rename = {c: f"recovered_{c}" for c in rec.columns if c not in {"number"}}
    rec = rec.rename(columns=rename)
    trials_df = trials_df.merge(rec, on="number", how="left")
    print(f"Merged {recovery_csv.name}  (+{len(rename)} columns from recovery)")

trials_csv = OPTUNA_DIR / "trials.csv"
trials_df.to_csv(trials_csv, index=False)
print(f"Wrote {trials_csv}  ({len(trials_df)} rows)")

# ------------------------------------------------------------------
# 2. Per-rung intermediate values -- Hyperband val_loss reports.
# ------------------------------------------------------------------
rows = [
    {"trial": t.number, "state": t.state.name, "rung": rung, "val_loss": val}
    for t in study.get_trials(deepcopy=False)
    for rung, val in t.intermediate_values.items()
]
intermediate_df = pd.DataFrame(rows).sort_values(["trial", "rung"]).reset_index(drop=True)
intermediate_csv = OPTUNA_DIR / "trials_intermediate.csv"
intermediate_df.to_csv(intermediate_csv, index=False)
print(
    f"Wrote {intermediate_csv}  "
    f"({len(intermediate_df)} reports across "
    f"{intermediate_df['trial'].nunique() if not intermediate_df.empty else 0} trials)"
)

# ------------------------------------------------------------------
# 3. Best-trial params -- consumed by the final retrain cell below.
# ------------------------------------------------------------------
best = study.best_trial
best_payload = {
    "study_name": STUDY_NAME,
    "storage": STORAGE,
    "trial_number": best.number,
    "value": float(best.value),
    "params": best.params,
    "datetime_start": str(best.datetime_start) if best.datetime_start else None,
    "datetime_complete": str(best.datetime_complete) if best.datetime_complete else None,
    "duration_seconds": (
        (best.datetime_complete - best.datetime_start).total_seconds()
        if best.datetime_complete and best.datetime_start
        else None
    ),
}
best_path = OPTUNA_DIR / "best_params.json"
best_path.write_text(json.dumps(best_payload, indent=2, default=str))
print(f"Wrote {best_path}")

# ------------------------------------------------------------------
# 4. Plotly HTML reports -- best-effort (some need >= 2 completed trials,
#    `plot_param_importances` needs scikit-learn).
# ------------------------------------------------------------------
plots = {
    "history": ov.plot_optimization_history,
    "intermediate_values": ov.plot_intermediate_values,
    "parallel_coordinate": ov.plot_parallel_coordinate,
    "slice": ov.plot_slice,
    "param_importances": ov.plot_param_importances,
}
for name, fn in plots.items():
    try:
        fig = fn(study)
        out_path = OPTUNA_DIR / f"{name}.html"
        fig.write_html(out_path)
        print(f"Wrote {out_path}")
    except Exception as exc:
        print(f"Skipped {name}.html  ({type(exc).__name__}: {exc})")

# ------------------------------------------------------------------
# 5. Console summary.
# ------------------------------------------------------------------
states = Counter(t.state.name for t in study.trials)
print("\nStudy summary:")
print(
    f"  trials:    {sum(states.values())} total  |  "
    + "  ".join(f"{k}: {v}" for k, v in states.items())
)
print(f"\n  best trial #{best.number}: value={best.value:.4f}")
for k, v in best.params.items():
    print(f"    {k}: {v}")

# Coalesce native (`user_attrs_val_loss_proto`, written during the trial)
# with recovered (`recovered_val_loss_proto`, from sidecar) so `top 5` shows
# the breakdown regardless of which path provided it.
for k in ("val_loss_proto", "val_loss_seq",
          "val_acc_proto_top1", "val_acc_proto_top5", "val_acc_seq"):
    native = f"user_attrs_{k}"
    recovered = f"recovered_{k}"
    if native in trials_df.columns and recovered in trials_df.columns:
        trials_df[k] = trials_df[native].combine_first(trials_df[recovered])
    elif native in trials_df.columns:
        trials_df[k] = trials_df[native]
    elif recovered in trials_df.columns:
        trials_df[k] = trials_df[recovered]

completed = trials_df[trials_df["state"] == "COMPLETE"].nsmallest(5, "value")
preferred_cols = [
    "number", "value",
    "val_loss_proto", "val_loss_seq",
    "val_acc_proto_top1", "val_acc_proto_top5", "val_acc_seq",
    "duration_seconds",
]
keep_cols = [c for c in preferred_cols if c in completed.columns]
keep_cols += [c for c in completed.columns if c.startswith("params_")]
print("\n  top 5 completed trials by val_loss:")
print(completed[keep_cols].to_string(index=False))

Wrote output/optuna/trials.csv  (39 rows)
Wrote output/optuna/trials_intermediate.csv  (108 reports across 39 trials)
Wrote output/optuna/best_params.json
Wrote output/optuna/history.html
Wrote output/optuna/intermediate_values.html
Wrote output/optuna/parallel_coordinate.html
Wrote output/optuna/slice.html
Wrote output/optuna/param_importances.html

Study summary:
  trials:    39 total  |  COMPLETE: 16  PRUNED: 22  FAIL: 1

  best trial #31: value=7.9714
    d_model: 256
    nhead: 8
    num_layers: 4
    dim_feedforward: 1024
    dropout: 0.14523704718965844
    lr: 0.00013271480005364088
    temperature: 0.1891638758952293

  top 5 completed trials by val_loss:
 number    value  params_d_model  params_dim_feedforward  params_dropout  params_lr  params_nhead  params_num_layers  params_temperature  duration_seconds
     31 7.971409             256                    1024        0.145237   0.000133             8                  4            0.189164        450.812957
     37 7.974730 